# BEVPlace++ on KITTI: experiments

Runtime -> Change runtime type -> **T4 GPU** (or better).

Same layout and habits as `01_ndt_kitti00.ipynb`: set the experiment in cell 1 and run every cell from top to bottom;
every cell skips work that is already done.

- **BEV variant**: the BEV image parameters (official values by default). Built on the Azure VM
  (`METHOD=bev bash scripts/azure_build.sh`), saved in `MyDrive/lpr/bev/<variant>/` with `config.json`,
  one `imgs_<seq>.tar` and one stats file per sequence.
- **Run**: the training parameters. Saved in `MyDrive/lpr/runs/bev/<run>/` (`config.json`, `metrics.csv`,
  a checkpoint per epoch, `eval_<seq>.json`, `eval_<seq>_matches.npz`).
- **Official checkpoint** (section 5): the authors' trained network evaluated on our BEV images, as a reproduction
  check. Saved in `MyDrive/lpr/runs/bev/official_<variant>/`.
- Heading analyses: `MyDrive/lpr/analysis/heading/bev/<run>/`; section 10 compares them with NDT-Transformer.

## 1. Experiment settings

In [1]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
DRIVE = "/content/drive/MyDrive/lpr"
SEQ = "00"                                     # training sequence
EVAL_SEQS = ["00", "02", "05", "06", "08"]

# BEV variant: only what differs from the defaults (src/lpr/experiments.py, BEV_DEFAULTS):
#   half_size=40.0, z_abs=40.0, voxel=0.4, res=0.4, max_count=10, ground="none"
BEV_TAG = "baseline"
BEV_PARAMS = dict()                 # e.g. dict(ground="patchwork")

# Training run: only what differs from the defaults (BEV_TRAIN_DEFAULTS, the official values):
#   train_frames=3000, pos_thr=5.0, neg_thr=7.0, epochs=40, lr=1e-4, batch=4, negatives=10,
#   margin=0.3, rot_aug=True, seed=1024
RUN_TAG = "baseline"
TRAIN_PARAMS = dict(epochs=20)               # e.g. dict(epochs=20)

KITTI = "/content/kitti"            # poses (and a few raw scans for the check in 4b)

## 2. Drive, GPU, code, poses

In [2]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, subprocess, importlib
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

url = REPO_URL
try:  # private repo: store a token in Colab Secrets as GITHUB_TOKEN
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/content/lpr"):
    subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
else:
    subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
%cd /content/lpr
!git log --oneline -1
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt faiss-cpu
missing_poses = [s for s in EVAL_SEQS if not os.path.exists(f"{KITTI}/poses/{s}.txt")]
if missing_poses:
    !python scripts/download_kitti.py --out {KITTI} --seqs {" ".join(missing_poses)} --poses_only

sys.path.insert(0, "/content/lpr/src")
import lpr.experiments as ex
importlib.reload(ex)

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB
/content/lpr
d2c8f71 (HEAD -> main, origin/main, origin/HEAD) BEVPlace++: BEV images, training, evaluation; method folders ndt/bev
third_party/NDT-Transformer @ 51f90df
third_party/BEVPlace2 @ 2d0636d5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.9/249.9 kB 21.7 MB/s eta 0:00:00
poses + calib ...
poses -> /content/kitti/poses


<module 'lpr.experiments' from '/content/lpr/src/lpr/experiments.py'>

### 2b. One time only: older results into the current layout
Moves `runs/<run>` to `runs/<method>/<run>` (and the heading analyses likewise). Does nothing once done.

## 3. Names of the BEV variant and the run

In [3]:
VARIANT, BEV_FULL, bev_exists = ex.resolve_variant(DRIVE, "bev", BEV_PARAMS, BEV_TAG)
VDIR = f"{DRIVE}/bev/{VARIANT}"         # permanent copy on Drive
BEV = f"/content/bev/{VARIANT}"         # working copy on this machine

RUN, TRAIN_FULL, run_state = ex.resolve_run(DRIVE, dict(TRAIN_PARAMS, seq=SEQ), VARIANT, RUN_TAG, method="bev")
RUN_DIR = f"{DRIVE}/runs/bev/{RUN}"

print("BEV variant:", VARIANT, "(on Drive)" if bev_exists else "(new: build it on Azure, see below)")
print(json.dumps(BEV_FULL))
print("run:        ", RUN, f"({run_state})")
print(json.dumps(TRAIN_FULL))
defaults = ex.BEV_DEFAULTS
args = " ".join(f"--{k} {v}" for k, v in BEV_FULL.items() if v != defaults[k])
print("\nOn the Azure VM (inside tmux, in the repository folder):")
print(f'  git pull && METHOD=bev VARIANT={VARIANT}' + (f' ARGS="{args}"' if args else "") +
      ' bash scripts/azure_build.sh "<SAS URL>"')

BEV variant: 2026-10-05_baseline_7711ab (on Drive)
{"half_size": 40.0, "z_abs": 40.0, "voxel": 0.4, "res": 0.4, "max_count": 10, "ground": "none"}
run:         2026-10-05_baseline_4ac7d1 (resume)
{"seq": "00", "train_frames": 3000, "pos_thr": 5.0, "neg_thr": 7.0, "epochs": 20, "lr": 0.0001, "batch": 4, "negatives": 10, "margin": 0.3, "rot_aug": true, "seed": 1024}

On the Azure VM (inside tmux, in the repository folder):
  git pull && METHOD=bev VARIANT=2026-10-05_baseline_7711ab bash scripts/azure_build.sh "<SAS URL>"


## 4. BEV images: from Azure to Drive to this machine
Copies the tars of the variant from the blob container (SAS URL in Colab Secrets as `AZURE_SAS`, permissions: read, list)
into `VDIR`, then unpacks them into `BEV`.

In [5]:
SAS_URL = userdata.get("AZURE_SAS")
tar_of = lambda s: f"{VDIR}/{ex.archive_name('bev', s)}"
missing = [s for s in EVAL_SEQS if not os.path.exists(tar_of(s))]
if missing:
    res = subprocess.run(["python", "scripts/fetch_from_azure.py", "--sas_url", SAS_URL, "--method", "bev",
                          "--variant", VARIANT, "--seqs", *missing, "--drive", DRIVE],
                         capture_output=True, text=True, cwd="/content/lpr")
    print(res.stdout, res.stderr[-2000:])
for s in EVAL_SEQS:
    if os.path.exists(tar_of(s)) and not os.path.exists(f"{BEV}/{s}/000000.png"):
        os.makedirs(BEV, exist_ok=True)
        subprocess.run(["tar", "-xf", tar_of(s), "-C", BEV], check=True)
HAVE = [s for s in EVAL_SEQS if os.path.exists(f"{BEV}/{s}/000000.png")]
print("BEV images on this machine for:", {s: len(os.listdir(f"{BEV}/{s}")) for s in HAVE})
if SEQ not in HAVE:
    print(f"sequence {SEQ} missing: build the variant on Azure first (command printed in cell 3)")

  config_azure.json: 0 MB in 1 s
seq 00: downloading
  imgs_00.tar: 23 MB in 9 s
  stats_00.csv: 0 MB in 1 s
seq 02: downloading
  imgs_02.tar: 24 MB in 9 s
  stats_02.csv: 0 MB in 1 s
seq 05: downloading
  imgs_05.tar: 15 MB in 6 s
  stats_05.csv: 0 MB in 1 s
seq 06: downloading
  imgs_06.tar: 9 MB in 4 s
  stats_06.csv: 0 MB in 1 s
seq 08: downloading
  imgs_08.tar: 25 MB in 9 s
  stats_08.csv: 0 MB in 1 s
sequences in 2026-10-05_baseline_7711ab: ['00', '02', '05', '06', '08']
 
BEV images on this machine for: {'00': 4541, '02': 4661, '05': 2761, '06': 1101, '08': 4071}


### 4b. Optional check: our BEV images = the official script's
Downloads the first 3 scans of sequence 00 and compares `lpr.bev` with `third_party/BEVPlace2/datasets/gen_bev_images.py`
pixel by pixel (needs open3d, about 1 minute to install). Expected last line: `IDENTICAL`.

In [6]:
RUN_CHECK = True
if RUN_CHECK:
    !pip install -q open3d
    !python scripts/download_kitti.py --out /content/kitti_check --seqs 00 --first 3
    scans = " ".join(f"/content/kitti_check/sequences/00/velodyne/{i:06d}.bin" for i in range(3))
    !python tests/test_bev_matches_official.py --scans {scans}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 67.8 MB/s eta 0:00:00
poses + calib ...
reading the Velodyne zip index ...
sequence 00: 3 scans
downloaded 3 new scans in 0.3 min -> /content/kitti_check
[ WARN:0@2.793] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
scan 0: shape (201, 201), pixels differing 0, max diff 0, {'n_raw': 124668, 'n_points': 121557, 'n_voxels': 12300, 'occupied_px': 6194}
scan 1: shape (201, 201), pixels differing 0, max diff 0, {'n_raw': 124605, 'n_points': 121226, 'n_voxels': 12035, 'occupied_px': 6033}
scan 2: shape (201, 201), pixels differing 0, max diff 0, {'n_raw': 124478, 'n_points

## 5. Reproduction check: the official checkpoint on our BEV images
`third_party/BEVPlace2/runs/Aug08_10-17-29/model_best.pth.tar` (trained by the authors on KITTI 00, chosen by them on NCLT).
If our BEV images and protocol match theirs, its recall should be close to the paper's numbers.

In [7]:
import pandas as pd
OFFICIAL_CKPT = "/content/lpr/third_party/BEVPlace2/runs/Aug08_10-17-29/model_best.pth.tar"
OFF_DIR = f"{DRIVE}/runs/bev/official_{VARIANT}"
os.makedirs(OFF_DIR, exist_ok=True)
ex.write_json(f"{OFF_DIR}/config.json", {"run": os.path.basename(OFF_DIR), "method": "bev", "bev_variant": VARIANT,
              "status": "finished", "params": {"source": "official checkpoint Aug08_10-17-29/model_best.pth.tar"}})
if os.path.exists(f"{VDIR}/config.json"):
    !cp {VDIR}/config.json {OFF_DIR}/bev_config.json

def evaluate_all(ckpt, out_dir):
    results = {}
    for s in HAVE:
        out = f"{out_dir}/eval_{s}.json"
        if not os.path.exists(out):
            res = subprocess.run(["python", "scripts/eval_bev_kitti.py", "--ckpt", ckpt, "--bev_dir", BEV,
                                  "--kitti_root", KITTI, "--seq", s, "--out", out],
                                 capture_output=True, text=True, cwd="/content/lpr")
            if res.returncode != 0:
                print(f"{s}: failed\n", res.stderr[-2000:]); continue
        r = json.load(open(out))
        results[s] = {k: round(r[k], 2) for k in ("R@1", "R@5", "R@1%")} | {"queries": r["n_queries_with_positive"]}
    return pd.DataFrame(results).T

display(evaluate_all(OFFICIAL_CKPT, OFF_DIR))

,R@1,R@5,R@1%,queries
00,100.00,100.00,100.00,682.0
02,99.01,100.00,100.00,304.0
05,97.54,98.43,98.66,447.0
06,100.00,100.00,100.00,270.0
08,94.10,98.69,99.67,305.0


## 6. Optional: timing and memory test (20 iterations)
Seconds per iteration x 750 is roughly one epoch, plus the descriptor cache at the start of each epoch.
If a batch (4 x 12 images) does not fit in GPU memory, training switches to one tuple per forward pass by itself
(`split_forward` in `metrics.csv`).

In [11]:
t = TRAIN_FULL
train_args = (f"--bev_dir {BEV} --kitti_root {KITTI} --bev_variant {VARIANT} --seq {t['seq']} "
              f"--train_frames {t['train_frames']} --pos_thr {t['pos_thr']} --neg_thr {t['neg_thr']} "
              f"--epochs {t['epochs']} --lr {t['lr']} --batch {t['batch']} --negatives {t['negatives']} "
              f"--margin {t['margin']} --seed {t['seed']}" + ("" if t["rot_aug"] else " --no_rot_aug"))
RUN_SMOKE = False
if RUN_SMOKE:
    !rm -rf /content/smoke
    !python scripts/train_bev_kitti.py {train_args} --log_dir /content/smoke --epochs 1 --max_iters 20 --eval_every 0 2>&1 | grep -v Warning

## 7. Training (resumes automatically after a disconnect, also inside an epoch)

In [ ]:
print("run:", RUN_DIR, f"({run_state})")
!python scripts/train_bev_kitti.py {train_args} --log_dir {RUN_DIR} 2>&1 | grep -v Warning

run: /content/drive/MyDrive/lpr/runs/bev/2026-10-05_baseline_4ac7d1 (new)
  warnings.warn(
  warnings.warn(msg)
NetVLAD initialisation: k-means on local features of the training images
  done in 11 s
device cuda, parameters 1.36 M, training frames 3000, images 4541
epoch 1: descriptor cache for hard mining
CUDA out of memory with the full batch -> one tuple per forward pass
epoch 1 it 50/750 loss 0.2974 1.96s/it (~23 min left in epoch) peak GPU mem 13.6 GB
epoch 1 it 100/750 loss 0.2829 1.90s/it (~21 min left in epoch) peak GPU mem 13.6 GB
epoch 1 it 150/750 loss 0.2695 1.91s/it (~19 min left in epoch) peak GPU mem 13.6 GB
epoch 1 it 200/750 loss 0.2447 1.90s/it (~17 min left in epoch) peak GPU mem 13.6 GB
epoch 1 it 250/750 loss 0.2246 1.90s/it (~16 min left in epoch) peak GPU mem 13.6 GB
epoch 1 it 300/750 loss 0.2020 1.90s/it (~14 min left in epoch) peak GPU mem 13.6 GB
epoch 1 it 350/750 loss 0.1781 1.90s/it (~13 min left in epoch) peak GPU mem 13.6 GB
epoch 1 it 400/750 loss 0.168

## 8. Evaluation of the last epoch on every sequence, and the table of all runs

In [4]:
import matplotlib.pyplot as plt
last = sorted(f for f in os.listdir(RUN_DIR) if f.startswith("epoch_"))[-1]
print("checkpoint:", last)
display(evaluate_all(f"{RUN_DIR}/{last}", RUN_DIR))

m = pd.read_csv(f"{RUN_DIR}/metrics.csv").drop_duplicates("epoch", keep="last")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(m.epoch, m.train_loss, marker="o"); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
if "R@1" in m:
    ax[1].plot(m.epoch, m["R@1"], marker="o", label="R@1"); ax[1].plot(m.epoch, m["R@1%"], marker="o", label="R@1%")
ax[1].set_title(f"KITTI {SEQ} recall (%)  {RUN}"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

checkpoint: epoch_18.pth


NameError: name 'evaluate_all' is not defined

In [ ]:
!python scripts/summarize_runs.py --drive {DRIVE} > /dev/null
display(pd.read_csv(f"{DRIVE}/runs_summary.csv", dtype={"run": str}))

## 9. Query headings
Same analysis as for NDT-Transformer: Recall@1 of the queries that revisit a place in the same direction,
crossing, or in the opposite direction. For the trained run and for the official checkpoint.

In [ ]:
from IPython.display import Image
for name, mdir in [(RUN, RUN_DIR), (os.path.basename(OFF_DIR), OFF_DIR)]:
    if not os.path.isdir(mdir) or not any(f.endswith("_matches.npz") for f in os.listdir(mdir)):
        print(name, ": no evaluation yet"); continue
    head = f"{DRIVE}/analysis/heading/bev/{name}"
    res = subprocess.run(["python", "scripts/heading_analysis.py", "--kitti_root", KITTI, "--seqs", *HAVE,
                          "--out_dir", head, "--matches_dir", mdir], capture_output=True, text=True, cwd="/content/lpr")
    print(name); print(res.stdout if res.returncode == 0 else res.stderr[-2000:])
display(Image(f"{DRIVE}/analysis/heading/bev/{RUN}/heading_hist.png")) if os.path.exists(f"{DRIVE}/analysis/heading/bev/{RUN}/heading_hist.png") else None

## 10. NDT-Transformer vs BEVPlace++
Recall@1 per sequence and per heading group, for every run with a heading analysis.

In [ ]:
import glob
rows = []
for f in sorted(glob.glob(f"{DRIVE}/analysis/heading/*/*/heading_summary.csv")):
    method, run = f.split("/")[-3], f.split("/")[-2]
    d = pd.read_csv(f, dtype={"seq": str})
    for _, r in d.iterrows():
        rows.append({"method": method, "run": run, "seq": r["seq"].zfill(2),
                     **{c: r[c] for c in d.columns if c.startswith("R@1") or c.startswith("%")}})
cmp = pd.DataFrame(rows)
for s, g in cmp.groupby("seq"):
    print("KITTI", s); display(g.drop(columns="seq").set_index(["method", "run"]))